In [3]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

SILVER = "business_silver"
GOLD = "dim_business"

df_business = spark.table(SILVER)

df_new = (
    df_business.select(
        "business_id",
        "name",
        "city",
        "state",
        "stars",
        "review_count",
        "categories",
        "_ingest_ts"
    )
    .withColumn("is_current",F.lit(True))
    .withColumn("effective_from", F.current_timestamp())
    .withColumn("effective_to",F.lit(None).cast("timestamp"))
)

if not spark.catalog.tableExists(GOLD):
    (
        df_new.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD)
    )
    print(F"[INT] {GOLD} created")

else:
    delta = DeltaTable.forName(spark, GOLD)
    df_current = spark.table(GOLD).filter(F.col("is_current" == True))

    scd_cols = ["name", "city", "state", "stars", "review_count", "categories"]

    join_cond = "current.business_id = new.business_id"
    change_cond = "OR".join([f"current.{c} <> new.{c}" for c  in scd_cols])

    df_changed = (
        df_current.alias("current")
        .join(df_new.alias("new"), "business_id")
        .filter(change_cond)
        .select("current.business_id")
    )

    (
        delta.alias("t")
        .merge(
            df_changed.alias("s"),
            "t.business_id = s.business_id AND t.is_current = true"
        )
        .whenMatchedUpdate(set={
            "is_current": "false",
            "effective_to": "current_timestamp()"
        })
        .execute()
    )

    df_existing_ids = spark.table(GOLD).filter(F.col("is_current") == True).select("business_id")

    df_to_insert = (
        df_new.join(df_existing_ids, "business_id", "left_anti")
        .union(
            df_new.join(df_changed, "business_id")
        )
    )

    (
        df_to_insert.write
        .format("delta")
        .mode("append")
        .saveAsTable(GOLD)
    )

    print(f"[SCD2] {GOLD} updated. Changed: {df_changed.count()}, New: {df_new.join(df_existing_ids, 'business_id', 'left_anti').count()}")

mssparkutils.notebook.exit("SUCCESS")

StatementMeta(, 47a64505-6ded-404b-81fe-ff811141c09a, 5, Finished, Available, Finished, False)

[INT] dim_business created
ExitValue: SUCCESS